In [ ]:
!pip install --upgrade pip
!pip install -q kokoro>=0.9.2 soundfile speechbrain sarvamai
!apt-get -qq -y install espeak-ng > /dev/null 2>&1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.1.1 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
libcugraph-cu12 25.6.0 requires libraft-cu12==25.6.*, but you have libraft-cu12 25.2.0 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.0a1 which is incompatible.
pylibcugraph-cu12 25.6.0 requires pylibraft-cu12==25.6.*, but you have pylibraft-cu12 25.2.0 which is incompatible.
pylibcugraph-cu12 25.6.0 requires rmm-cu12==25.6.*, but you have rmm-cu12 25.2.0 which is incompatible.


In [26]:
# import sys
# from pathlib import Path
# import shutil

# def confirm_and_delete(dry_run=False):
#     cwd = Path.cwd()
#     items = list(cwd.iterdir())
#     if not items:
#         print("Directory is already empty.")
#         return

#     print(f"Current directory: {cwd}")
#     print(f"About to delete {len(items)} item(s):")
#     for p in items:
#         print(" -", p.name)

#     if dry_run:
#         print("\nDry run enabled. No files will be deleted.")
#         return

#     resp = input("\nType DELETE (all caps) to permanently remove these items: ").strip()
#     if resp != "DELETE":
#         print("Aborted.")
#         return

#     for p in items:
#         try:
#             if p.is_dir() and not p.is_symlink():
#                 shutil.rmtree(p)
#             else:
#                 p.unlink()
#         except Exception as e:
#             print(f"Failed to remove {p}: {e}")

#     print("Deletion complete.")

# if __name__ == "__main__":
#     # Pass "dry" as command-line arg to list items without deleting
#     dry = len(sys.argv) > 1 and sys.argv[1].lower() in ("dry", "--dry-run", "-n")
#     confirm_and_delete(dry_run=dry)

In [27]:
!git clone https://github.com/sathishkumar67/Kokoro-Indian-Voice.git
!mv Kokoro-Indian-Voice/* ./
!rm -rf Kokoro-Indian-Voice

Cloning into 'Kokoro-Indian-Voice'...
remote: Enumerating objects: 172, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 172 (delta 0), reused 56 (delta 0), pack-reused 112 (from 3)
Receiving objects: 100% (172/172), 347.46 MiB | 46.42 MiB/s, done.
Updating files: 100% (159/159), done.


In [ ]:
# Standard library imports
import os
import glob
import random

# PyTorch core and convenience modules
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Audio I/O and processing
import torchaudio
import soundfile as sf

# Pretrained models / project pipelines
from speechbrain.pretrained import EncoderClassifier
from kokoro import KPipeline

# Notebook display helpers
from IPython.display import display, Audio

# Device selection: prefer CUDA if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [28]:
# Load a pretrained speaker-recognition encoder (ECAPA-TDNN trained on VoxCeleb).
# We pass run options so the model runs on the selected device (GPU if available).
encoder = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb",
                                         run_opts={"device": device})

def get_speaker_emb(path):
    """
    Load an audio file, ensure it is sampled at 16 kHz, and return a speaker embedding.

    This function:
      - reads the file at `path` using torchaudio.load
      - resamples to 16 kHz if needed
      - moves the waveform to the model device
      - extracts an embedding from the pretrained EncoderClassifier without
        tracking gradients (inference mode)
      - moves the embedding to CPU and detaches it from the computation graph

    Args:
        path (str or Path): Path to an audio file readable by torchaudio.

    Returns:
        torch.Tensor: 1-D CPU tensor containing the speaker embedding. The
                      embedding dimensionality is model-specific (commonly 192).
    """
    # Load waveform (shape: [channels, samples]) and sample rate
    wav, sr = torchaudio.load(path)

    # If necessary, resample to 16 kHz — the pretrained encoder expects 16k audio.
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)

    # Move waveform to device (CPU or CUDA) before feeding into the model.
    wav = wav.to(device)

    # Run the encoder in inference mode to avoid storing gradients / extra memory.
    with torch.no_grad():
        # encode_batch handles variable-length inputs internally and returns
        # a batched embedding tensor.
        emb = encoder.encode_batch(wav)

    # Remove batch dimension if present, detach and move to CPU for downstream use.
    emb = emb.squeeze(0).detach().cpu()

    return emb # [1, 192]

In [44]:
kokoro_voice_paths = [f"kokoro_voices/{voice}" for voice in os.listdir("kokoro_voices")]
indian_voice_paths = [f"audio_samples/{voice}" for voice in os.listdir("audio_samples")]
kokoro_embs_list = []

In [45]:
for kokoro_voice_path in kokoro_voice_paths:
    kokoro_embs = torch.load(kokoro_voice_path, map_location="cpu").float().mean(dim=0).squeeze(0)  # [256] – mean across time
    kokoro_embs_list.append(kokoro_embs)

In [46]:
kokoro_embs_list = torch.stack(kokoro_embs_list)  # [N, 256]
print("Loaded Kokoro voices:", kokoro_embs_list.shape)

indian_embs_list = torch.stack([get_speaker_emb(indian_voice_path) for indian_voice_path in indian_voice_paths])  # [N, 256]
print("Loaded Indian voices:", indian_embs_list.shape)

Loaded Kokoro voices: torch.Size([54, 256])
Loaded Indian voices: torch.Size([100, 1, 192])


In [47]:
class Adapter(nn.Module):
    def __init__(self):
        super(Adapter, self).__init__()
        self.fc1 = nn.Linear(192, 512)
        self.fc2 = nn.Linear(512, 512)
        self.fc3 = nn.Linear(512, 192)
        self.fc4 = nn.Linear(192, 256)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x, self.fc4(x)

In [48]:
adapter = Adapter()
adapter.to(device)
print("Loaded adapter")

Loaded adapter


In [72]:
learning_rate = 5e-5       # small learning rate
weight_decay = 1e-3       # L2 regularization
epochs = 2000              # max epochs
patience = 100             # early stopping patience
batch_size = 4
best_loss = float('inf')
epochs_no_improve = 0
best_state = None
kokoro_voice_sim_weight, indian_voice_sim_weight = 0.3, 1.0

In [67]:
optimizer = optim.AdamW(adapter.parameters(), lr=learning_rate, weight_decay=weight_decay, fused=True)

In [ ]:
for epoch in range(epochs):
    adapter.train()
    total_loss = 0.0

    # shuffle data at the start of each epoch
    perm = torch.randperm(len(indian_embs_list))
    for i in range(0, len(indian_embs_list), batch_size):
        kokoro_indices = random.sample(range(len(kokoro_embs_list)), 4)
        optimizer.zero_grad()

        batch_indices = perm[i:i + batch_size]
        indian_batch = indian_embs_list[batch_indices].to(device) # [4, 192]
        kokoro_batch = kokoro_embs_list[kokoro_indices].to(device) # [4, 256]

        indian_dim_vector, kokoro_dim_vector = adapter(indian_batch) # [4, 192], [4, 256]
        
        loss = (1 - F.cosine_similarity(kokoro_dim_vector, kokoro_batch, dim=1).mean()) * kokoro_voice_sim_weight + (1 - F.cosine_similarity(indian_dim_vector, indian_batch, dim=1).mean()) * indian_voice_sim_weight
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(indian_embs_list)
    print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        best_state = adapter.state_dict()
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("Early stopping triggered.")
            break

In [ ]:
def make_kokoro_voice(wav_path, adapter, output_path="indian_english.pt"):
    emb = get_speaker_emb(wav_path).unsqueeze(0).to(device)  # [1,192]
    with torch.no_grad():
        _, mapped = adapter(emb) 
        mapped = mapped.cpu()  # [1,256]
    if mapped.dim() == 1:
        mapped = mapped.unsqueeze(0)
    # Now shape = [1, 256]
    mapped = mapped.repeat(510, 1, 1)  # [510,1,256]
    torch.save(mapped, output_path)
    print(f"✅ Saved Kokoro-compatible voice: {output_path}")

make_kokoro_voice("audio_samples/1.wav", adapter, "indian_english.pt")

torch.Size([1, 1, 256])
torch.Size([1, 1, 256])
torch.Size([510, 1, 256])
✅ Saved Kokoro-compatible voice: indian_english.pt


In [81]:
import torch
from kokoro import KPipeline
from IPython.display import display, Audio
pipeline = KPipeline(lang_code='a')
text = '''
Kokoro is an open-weight TTS model with 82 million parameters. Despite its lightweight architecture, it delivers comparable quality to larger models while being significantly faster and more cost-efficient. With Apache-licensed weights, [Kokoro](/kˈOkəɹO/) can be deployed anywhere from production environments to personal projects.
'''
voice = torch.load("indian_english.pt", map_location="cpu")
generator = pipeline(text, voice=voice, speed=1.0, split_pattern=r'\n+')

for i, (gs, ps, audio) in enumerate(generator):
    print(i, gs)
    display(Audio(data=audio, rate=24000, autoplay=i==0))

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


kokoro-v1_0.pth:   0%|          | 0.00/327M [00:00<?, ?B/s]

0 Kokoro is an open-weight TTS model with 82 million parameters. Despite its lightweight architecture, it delivers comparable quality to larger models while being significantly faster and more cost-efficient. With Apache-licensed weights, Kokoro can be deployed anywhere from production environments to personal projects.


/usr/local/lib/python3.11/dist-packages/IPython/lib/display.py:175: RuntimeWarning: invalid value encountered in cast
  return scaled.astype("<h").tobytes(), nchan
